In [7]:
import numpy as np
import gymnasium as gym
import math
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as patches
import os.path as path
from shutil import copyfile
import pickle
import time
import itertools

In [8]:

# This section defines the core simulation parameters for the reinforcement learning environment.
# These values control the length of an episode, the number of episodes, and the dynamics of the simulated process.
epLen = 10 # H: Defines the maximum number of steps or time stages within a single simulation episode.
nEps = 2000 # K: Specifies the total number of episodes (or learning iterations) the agent will undergo.
numIters =1 # The iteration number, potentially useful for future extensions or multi-agent scenarios.
starting_state=4 # The initial state value from which each episode begins.
drift_bias=0.05 # The drift parameter in the Ornstein-Uhlenbeck (O-U) process, influencing the tendency of the state to return to a mean.
drift_state_coef = -0.1 # The action parameter in the drift term, representing how the agent's action influences the state's drift.
drift_action_coef = 0.01
sigma=0.1 # The volatility parameter in the O-U process, determining the magnitude of random fluctuations in the state.
Delta=1 # The step size or time increment for each step within an episode, often representing a discrete time unit.
action_dim = 1  # This sets the dimensionality of the action space. The agent will output a vector of this size.
state_dim = 1

# This section defines hyperparameters for the Adaptive Model-Based Discretization algorithm.
# These parameters often require tuning to achieve optimal performance and convergence.
initial_q=1837.1 # The initial optimistic Q-value and V-value estimation used for unvisited states or nodes.
rho=10 # The initial radius for state space partitioning, defining the size of the initial 'balls' or regions.
rho_1=0.5 # The initial radius for action space partitioning, defining the size of action regions within each state ball.
C_h = 5 # The Lipschitz constant, used in the Bellman update to bound the growth of the value function.
D = 10 * np.sqrt(2)
split_threshold=2 # A constant that determines when a node (state-action region) should be split, based on the number of visits.
scaling=0.01 # A scaling constant for the UCB (Upper Confidence Bound) bonus term, balancing exploration and exploitation.
out_of_bounds_q = -505


In [9]:

class BellmanSolverScalar:
    def __init__(self,
                 epLen,
                 Delta,
                 rho,
                 sigma,
                 drift_bias,
                 drift_state_coef,
                 drift_action_coef,
                 out_of_bounds_q,
                 starting_state,
                 n_state_grid=801,
                 n_action_grid=401,
                 n_quadrature=81):
        self.epLen = epLen
        self.Delta = Delta
        self.sqrt_Delta = math.sqrt(Delta)
        self.rho = rho
        self.sigma = sigma
        self.drift_bias = drift_bias
        self.drift_state_coef = drift_state_coef
        self.drift_action_coef = drift_action_coef
        self.out_of_bounds_q = out_of_bounds_q
        self.starting_state = starting_state

        self.n_state_grid = n_state_grid
        self.n_action_grid = n_action_grid
        self.n_quadrature = n_quadrature

        self.state_grid = np.linspace(-rho, rho, n_state_grid)
        self.action_grid = np.linspace(0.0, 10.0, n_action_grid)

        pts, wts = np.polynomial.hermite.hermgauss(n_quadrature)
        self.quad_z = pts * np.sqrt(2.0)
        self.quad_w = wts / np.sqrt(np.pi)

        self.V = np.zeros((epLen + 1, n_state_grid))
        self.policy = np.zeros((epLen, n_state_grid))
        self.solved = False

    def reward_mean(self, x, a):
        return (x - a) ** 2

    def next_state(self, x, a, z):
        drift = self.drift_bias + self.drift_state_coef * x + self.drift_action_coef * a
        return x + drift * self.Delta + self.sigma * self.sqrt_Delta * z

    def expected_backup(self, x, a, h):
        total = 0.0
        for z, w in zip(self.quad_z, self.quad_w):
            x_next = self.next_state(x, a, z)

            if abs(x_next) > self.rho:
                val = self.out_of_bounds_q
            else:
                stage = self.reward_mean(x, a)
                cont = np.interp(x_next, self.state_grid, self.V[h + 1])
                val = stage + cont

            total += w * val
        return total

    def solve(self):
        self.V[self.epLen, :] = 0.0

        for h in range(self.epLen - 1, -1, -1):
            for i, x in enumerate(self.state_grid):
                q_vals = np.array([self.expected_backup(x, a, h) for a in self.action_grid])
                best_idx = np.argmax(q_vals)
                self.V[h, i] = q_vals[best_idx]
                self.policy[h, i] = self.action_grid[best_idx]

        self.solved = True

    def get_value(self, x, h):
        return float(np.interp(x, self.state_grid, self.V[h]))

    def get_optimal_action(self, x, h):
        idx = np.argmin(np.abs(self.state_grid - x))
        return float(self.policy[h, idx])
 

In [16]:


class Experiment(object):

    def __init__(self, env, agent_list, dict):
        assert isinstance(env, Environment), "Provided environment must be an instance of the Environment class."

        self.seed = dict['seed']
        self.epFreq = dict['recFreq']
        self.targetPath = dict['targetPath']
        self.deBug = dict['deBug']
        self.nEps = dict['nEps']
        self.env = env
        self.epLen = env.get_epLen()
        self.num_iters = dict['numIters']
        self.agent_list = agent_list
        self.data = np.zeros([dict['nEps'] * self.num_iters, 4])

        np.random.seed(self.seed)

    def run_2(self):
    
        print('Running experiment')

        for i in range(self.num_iters):
            agent = self.agent_list[i]

            for ep in range(1, self.nEps + 1):
                self.env.reset()
                oldState = self.env.state
                epReward = 0
                agent.update_policy(ep)
                pContinue = 1
                h = 0

                while pContinue > 0 and h < self.env.epLen:
                    if self.deBug:
                        print('state : ' + str(oldState))

                    action = agent.pick_action(oldState, h)

                    if self.deBug:
                        print('action : ' + str(action))

                    reward, newState, pContinue = self.env.advance(action)
                    epReward += reward

                    agent.update_obs_2(oldState, action, reward, newState, h)

                    oldState = newState
                    h += 1

                if self.deBug:
                    print('final state: ' + str(newState))
                    print('Total Reward: ' + str(epReward))

                index = ep - 1
                self.data[index, 0] = ep - 1
                self.data[index, 1] = i
                self.data[index, 2] = epReward
                self.data[index, 3] = agent.get_num_arms()

     
        print('Experiment complete')


    def save_data(self):

        print('Saving data')


        dt = pd.DataFrame(self.data, columns=['episode', 'iteration', 'epReward', 'Number_of_Balls'])
        dt = dt[(dt.T != 0).any()]
        return dt


class Node_2():
    def __init__(self, qVal, rEst, muEst, sigmaEst, num_visits, num_unique_visits,
                 num_splits, state_val, action_val, radius, action_radius):
        self.qVal = qVal
        self.rEst = rEst
        self.muEst = muEst
        self.sigmaEst = sigmaEst
        self.num_visits = num_visits
        self.num_unique_visits = num_unique_visits
        self.num_splits = num_splits
        self.state_val = state_val
        self.action_val = float(action_val)
        self.radius = radius
        self.action_radius = action_radius
        self.children = None

    def split_node_2(self, flag, epLen):
        half_radius = self.radius * 0.5
        half_action_radius = self.action_radius * 0.5
        state_val = self.state_val
        action_val = self.action_val
        num_splits_plus1 = self.num_splits + 1
        low_visits = self.num_visits <= 1

        state_offsets = [-1, 1]
        action_offsets = [-1, 1]
        children = []

        for s_off in state_offsets:
            new_state = float(state_val + s_off * half_radius)

            for a_off in action_offsets:
                new_action = float(action_val + a_off * half_action_radius)
                new_action = float(np.clip(new_action, 0, 10))

                if low_visits:
                    child = Node_2(
                        initial_q, 0, 0, 0,
                        self.num_visits, 0, num_splits_plus1,
                        new_state, new_action,
                        half_radius, half_action_radius
                    )
                else:
                    child = Node_2(
                        self.qVal, self.rEst, self.muEst, self.sigmaEst,
                        self.num_visits, self.num_visits, num_splits_plus1,
                        new_state, new_action,
                        half_radius, half_action_radius
                    )

                children.append(child)

        self.children = children
        return self.children


class Tree_2():
    def __init__(self, epLen, flag):
        self.epLen = epLen
        self.flag = flag
        self.flag_scale_2 = scaling

        self.head_1 = Node_2(initial_q, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5)
        self.head_2 = Node_2(initial_q, 0, 0, 0, 0, 0, 0, -5, 5, 5, 5)

        self.state_leaves = [self.head_1.state_val, self.head_2.state_val]
        self.vEst = [1837.1, 1837.1]
        self.tree_leaves = [self.head_1, self.head_2]

    def get_head(self):
        return self.head_1, self.head_2

    def block_diameter_2(self, node):
        return float(np.sqrt(node.radius**2 + node.action_radius**2))

    def confidence_radius_2(self, node):
        n = max(1, node.num_unique_visits)
        local_scale = 1.0 + abs(node.muEst) + np.sqrt(max(node.sigmaEst, 0.0))
        return float(self.flag_scale_2 * local_scale * np.sqrt(np.log(n + 2.0) / n))

    def should_split_2(self, node):
        if node.num_unique_visits < 2:
            return False

        conf = self.confidence_radius_2(node)
        diam = self.block_diameter_2(node)
        return conf <= diam

    def split_node_2(self, node, timestep, previous_tree):
        children = node.split_node_2(self.flag, self.epLen)

        self.tree_leaves.remove(node)
        for child in children:
            self.tree_leaves.append(child)

        child_1_state = children[0].state_val
        child_1_radius = children[0].radius

        if np.min(np.abs(np.asarray(self.state_leaves) - child_1_state)) >= child_1_radius:
            parent = node.state_val
            parent_index = self.state_leaves.index(parent)
            parent_vEst = self.vEst[parent_index]

            self.state_leaves.pop(parent_index)
            self.vEst.pop(parent_index)

            num_action_offsets = 2 ** action_dim
            self.state_leaves.append(children[0].state_val)
            self.state_leaves.append(children[num_action_offsets].state_val)
            self.vEst.append(parent_vEst)
            self.vEst.append(parent_vEst)

        return children

    def get_num_balls(self, node):
        if node.children is None:
            return 1
        num_balls = 0
        for child in node.children:
            num_balls += self.get_num_balls(child)
        return num_balls

    def get_number_of_active_balls(self):
        return self.get_num_balls(self.head_1) + self.get_num_balls(self.head_2)

    def get_active_ball_recursion(self, state, node):
        if node.children is None:
            return node, node.qVal
        else:
            active_node = None
            qVal = -np.inf

            for child in node.children:
                if self.state_within_node(state, child):
                    new_node, new_qVal = self.get_active_ball_recursion(state, child)
                    if new_qVal >= qVal:
                        active_node, qVal = new_node, new_qVal

            if active_node is None:
                return node, node.qVal

            return active_node, qVal

    def get_active_ball(self, state):
        safe_state = float(np.clip(state, -rho, rho))
        if safe_state >= 0:
            active_node, qVal = self.get_active_ball_recursion(safe_state, self.head_1)
        else:
            active_node, qVal = self.get_active_ball_recursion(safe_state, self.head_2)
        return active_node, qVal

    def state_within_node(self, state, node):
        return np.abs(state - node.state_val) <= node.radius


class AdaptiveModelBasedDiscretization_2(Agent):
    def __init__(self, epLen, numIters, scaling, split_threshold, inherit_flag, flag):
        self.epLen = epLen
        self.numIters = numIters
        self.scaling = scaling
        self.split_threshold = split_threshold
        self.inherit_flag = inherit_flag
        self.flag = flag
        self.tree_list = []

        for _ in range(epLen):
            tree = Tree_2(epLen, self.inherit_flag)
            self.tree_list.append(tree)

    def reset(self):
        self.tree_list = []
        for _ in range(self.epLen):
            tree = Tree_2(self.epLen, self.inherit_flag)
            self.tree_list.append(tree)

    def get_num_arms(self):
        total_size = 0
        for tree in self.tree_list:
            total_size += tree.get_number_of_active_balls()
        return total_size

    def update_obs_2(self, obs, action, reward, newObs, timestep):
        tree = self.tree_list[timestep]
        active_node, _ = tree.get_active_ball(obs)

        active_node.num_visits += 1
        active_node.num_unique_visits += 1
        t = active_node.num_unique_visits

        active_node.rEst = ((t - 1) * active_node.rEst + reward) / t

        if timestep != self.epLen - 1:
            delta_state = newObs - obs
            old_mu = active_node.muEst
            active_node.muEst = ((t - 1) * active_node.muEst + delta_state) / t
            active_node.sigmaEst = ((t - 1) * active_node.sigmaEst + (delta_state - old_mu) ** 2) / t

        if self.flag is False:
            if timestep == self.epLen - 1:
                active_node.qVal = min(
                    active_node.qVal,
                    initial_q,
                    active_node.rEst + self.scaling / np.sqrt(active_node.num_visits) + self.scaling * active_node.radius
                )
            else:
                next_tree = self.tree_list[timestep + 1]
                vEst = min(next_tree.vEst) + C_h * (1 + active_node.muEst ** 2 + active_node.sigmaEst ** 2)
                active_node.qVal = min(
                    active_node.qVal,
                    initial_q,
                    active_node.rEst + vEst + self.scaling / np.sqrt(active_node.num_visits) + self.scaling * active_node.radius
                )

            index = 0
            for state_val in tree.state_leaves:
                _, qMax = tree.get_active_ball(state_val)
                tree.vEst[index] = min(qMax, initial_q, tree.vEst[index])
                index += 1

        if tree.should_split_2(active_node):
            if timestep >= 1:
                _ = tree.split_node_2(active_node, timestep, self.tree_list[timestep - 1])
            else:
                _ = tree.split_node_2(active_node, timestep, None)

    def update_policy(self, k):
        if self.flag:
            for h in np.arange(self.epLen - 1, -1, -1):
                tree = self.tree_list[h]
                for node in tree.tree_leaves:
                    if node.num_unique_visits == 0:
                        node.qVal = initial_q
                    else:
                        if h == self.epLen - 1:
                            node.qVal = min(node.qVal, initial_q, node.rEst + self.scaling / np.sqrt(node.num_visits))
                        else:
                            next_tree = self.tree_list[h + 1]
                            vEst = min(next_tree.vEst) + C_h * (1 + node.muEst ** 2 + node.sigmaEst ** 2)
                            node.qVal = min(node.qVal, initial_q, node.rEst + vEst + self.scaling / np.sqrt(node.num_visits))

                index = 0
                for state_val in tree.state_leaves:
                    _, qMax = tree.get_active_ball(state_val)
                    tree.vEst[index] = min(qMax, initial_q, tree.vEst[index])
                    index += 1

    def split_ball(self, node):
        children = node.split_ball()
        return children

    def greedy(self, state, timestep, epsilon=0):
        tree = self.tree_list[timestep]
        active_node, _ = tree.get_active_ball(state)

        action = np.random.uniform(
            active_node.action_val - active_node.action_radius,
            active_node.action_val + active_node.action_radius
        )
        return float(np.clip(action, 0, 10))

    def pick_action(self, state, timestep):
        action = self.greedy(state, timestep)
        return action

NameError: name 'Agent' is not defined

In [15]:
class AdaDiffEnvironment(Environment):
    """A custom environment simulating an adaptive diffusion process,
    where the state evolves based on drift, action, and stochastic volatility."""
    def __init__(self, epLen, starting_state):
        """
        Initializes the Adaptive Diffusion Environment.
        epLen: The maximum number of steps per episode.
        starting_state: The initial state for each episode.
        """
        self.epLen = epLen
        self.state = starting_state # Current state of the process.
        self.starting_state = starting_state # Stores the fixed starting state.
        self.timestep = 0 # Tracks the current step within an episode.


    def get_epLen(self):
        """Returns the maximum episode length."""
        return self.epLen

    def reset(self):
        """Resets the environment to its starting state for a new episode."""
        self.timestep = 0
        self.state = self.starting_state

    def advance(self, action):
        """Simulates one step of the adaptive diffusion process.
        The state evolves based on an Ornstein-Uhlenbeck-like dynamic, influenced by the action.

        Args:
            action: A multi-dimensional action vector from the agent.

        Returns:
            reward: The reward received for this step.
            new_state: The state after the transition.
            pContinue: 0 if episode ends, 1 if it continues.
        """

        noise = float(np.random.randn())
        state = float(self.state)
        action = float(np.clip(action, 0, 10))
        
        drift = drift_bias + drift_state_coef * state + drift_action_coef * action
        new_state = state + drift * Delta + sigma * np.sqrt(Delta) * noise


        if abs(new_state) > rho:
            reward = out_of_bounds_q
            pContinue = 0
            new_state = float(np.clip(new_state, -rho, rho))
        else:
            reward = float(np.random.normal(loc=(state - action)**2, scale=0.1))
            pContinue = 1

        self.state = new_state # Updates the environment's current state.
        self.timestep += 1 # Increments the internal timestep counter.

        pContinue = 1
        # If the episode length is reached, the episode terminates.
        if self.timestep == self.epLen:
            pContinue = 0

        return reward, new_state, pContinue



NameError: name 'Environment' is not defined

In [10]:
from joblib import Parallel, delayed

# Helper function to create an instance of the Adaptive Diffusion Environment.
def make_diffMDP(epLen, starting_state):
    return AdaDiffEnvironment(epLen, starting_state)

# Defines a function to run a single iteration of the experiment.
# This function will be executed in parallel across multiple processes.
def run_single_experiment_iteration(iteration_seed):
    # Re-create the environment for each parallel iteration to ensure independent simulations.
    env_single = make_diffMDP(epLen, starting_state)

    # Re-create the agent for each iteration. This ensures each agent starts fresh
    # and is trained on its own separate environment run.
    agent_single = AdaptiveModelBasedDiscretization(epLen, nEps, scaling, split_threshold, False, False)

    # Dictionary containing configuration for this single experiment run.
    # numIters is set to 1 because each parallel job runs one agent's simulation.
    dictionary_single = {'seed': iteration_seed, 'epFreq' : 1, 'targetPath': './tmp_iter_{}.csv'.format(iteration_seed), 'deBug' : False, 'nEps': nEps, 'recFreq' : 10, 'numIters' : 1}

    # Initialize the Experiment class with the single environment and agent.
    exp_single = Experiment(env_single, [agent_single], dictionary_single)

    # Run the experiment simulation for this iteration.
    exp_single.run()

    # Save the data generated by this single experiment run and extract the episode rewards.
    dt_data_single = exp_single.save_data()
    return dt_data_single.epReward

In [11]:

def run_single_experiment_with_agent(iteration_seed):
    env_single = make_diffMDP(epLen, starting_state)
    agent_single = AdaptiveModelBasedDiscretization(
        epLen, nEps, scaling, split_threshold, False, False
    )

    dictionary_single = {
        'seed': iteration_seed,
        'epFreq': 1,
        'targetPath': f'./tmp_iter_{iteration_seed}.csv',
        'deBug': False,
        'nEps': nEps,
        'recFreq': 10,
        'numIters': 1
    }

    exp_single = Experiment(env_single, [agent_single], dictionary_single)
    exp_single.run()
    dt_data_single = exp_single.save_data()
    return dt_data_single, agent_single


def plot_q_partition_heatmap(tree, timestep_label=""):
    fig, ax = plt.subplots(figsize=(8, 5))

    q_vals = [node.qVal for node in tree.tree_leaves]
    q_min, q_max = min(q_vals), max(q_vals)

    for node in tree.tree_leaves:
        x0 = node.state_val - node.radius
        y0 = node.action_val - node.action_radius
        width = 2 * node.radius
        height = 2 * node.action_radius

        q_norm = 0.5 if q_max == q_min else (node.qVal - q_min) / (q_max - q_min)
        color = plt.cm.RdYlGn_r(q_norm)

        rect = patches.Rectangle(
            (x0, y0), width, height,
            linewidth=1, edgecolor='black', facecolor=color
        )
        ax.add_patch(rect)

    ax.set_xlim(-rho, rho)
    ax.set_ylim(0, 10)
    ax.set_xlabel("State Space")
    ax.set_ylabel("Action Space")
    ax.set_title(f"Heat Map of Q Values {timestep_label}")
    plt.grid(False)
    plt.show()
    
    
def plot_learning_curve_with_true_value(vpi_estimate, true_value):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(vpi_estimate) + 1), vpi_estimate, label='estimated VPI')
    plt.axhline(y=true_value, color='red', linestyle='-', label='optimal value')
    plt.xlabel("Episode")
    plt.ylabel("Episode reward")
    plt.title("Estimated VPI vs episode")
    plt.legend()
    plt.grid(True)
    plt.show()
    
    


def plot_log_regret(vpi_estimate, true_value, fit_start=1000):
    episodes = np.arange(1, len(vpi_estimate) + 1)
    regret = np.maximum(true_value - np.asarray(vpi_estimate), 1e-10)
    cumulative_regret = np.cumsum(regret)

    valid_idx = episodes >= fit_start
    x = np.log(episodes[valid_idx])
    y = np.log(cumulative_regret[valid_idx])

    slope, intercept = np.polyfit(x, y, 1)
    y_fit = slope * x + intercept

    plt.figure(figsize=(10, 6))
    plt.plot(x, y, label='log regret')
    plt.plot(x, y_fit, 'r-', label=f'Linear Regression (Slope {slope:.3f})')
    plt.xlabel("log episode")
    plt.ylabel("log regret")
    plt.title(f"Log Regret vs Log Episode with Linear Fit Episode {fit_start}-{len(vpi_estimate)}")
    plt.legend()
    plt.grid(True)
    plt.show()

    return slope



In [12]:

def run_single_experiment_with_agent_2(iteration_seed):
    env_single = make_diffMDP(epLen, starting_state)

    agent_single = AdaptiveModelBasedDiscretization_2(
        epLen, nEps, scaling, split_threshold, False, False
    )

    dictionary_single = {
        'seed': iteration_seed,
        'epFreq': 1,
        'targetPath': f'./tmp_iter_{iteration_seed}.csv',
        'deBug': False,
        'nEps': nEps,
        'recFreq': 10,
        'numIters': 1
    }

    exp_single = Experiment(env_single, [agent_single], dictionary_single)
    exp_single.run_2()
    dt_data_single = exp_single.save_data()

    return dt_data_single, agent_single


def get_ep_rewards(iteration_seed):
    dt_data_single, _ = run_single_experiment_with_agent_2(iteration_seed)
    return dt_data_single.epReward


start = time.perf_counter()

# # Solve Bellman benchmark
# solver = BellmanSolverScalar(
#     epLen=epLen,
#     Delta=Delta,
#     rho=rho,
#     sigma=sigma,
#     drift_bias=drift_bias,
#     drift_state_coef=drift_state_coef,
#     drift_action_coef=drift_action_coef,
#     out_of_bounds_q=out_of_bounds_q,
#     starting_state=starting_state
# )
# solver.solve()
# true_value = solver.get_value(starting_state, 0)

# Train one agent and keep it for partition plotting
dt_single, trained_agent = run_single_experiment_with_agent_2(123)

# Plot learned partition
plot_q_partition_heatmap(
    trained_agent.tree_list[9],
    timestep_label=r"for $P_9^{2000}$"
)

# Run multiple experiments and average episode rewards
n = 10
list_of_vpi_2 = Parallel(n_jobs=-1)(
    delayed(get_ep_rewards)(i) for i in range(n)
)

vpi_df = pd.DataFrame(list_of_vpi_2).T
vpi_estimate = vpi_df.mean(axis=1).values

# Plot learning curve against Bellman benchmark
plot_learning_curve_with_true_value(vpi_estimate, true_value)

# Plot log regret and estimate slope
slope = plot_log_regret(vpi_estimate, true_value, fit_start=1000)

print("Estimated regret slope:", slope)
print("True Bellman value at x0:", true_value)

end = time.perf_counter()
print(f"Elapsed time: {end - start:.4f} seconds")

NameError: name 'AdaDiffEnvironment' is not defined

In [ ]:
start = time.perf_counter()

# Solve Bellman benchmark
solver = BellmanSolverScalar(
    epLen=epLen,
    Delta=Delta,
    rho=rho,
    sigma=sigma,
    drift_bias=drift_bias,
    drift_state_coef=drift_state_coef,
    drift_action_coef=drift_action_coef,
    out_of_bounds_q=out_of_bounds_q,
    starting_state=starting_state
)
solver.solve()
true_value = solver.get_value(starting_state, 0)

# Train one agent and keep tree
dt_single, trained_agent = run_single_experiment_with_agent(123)

# Top plot: learned partition at last timestep, e.g. h=9
plot_q_partition_heatmap(trained_agent.tree_list[9], timestep_label=r"for $P_9^{2000}$")

In [ ]:
class Experiment(object):

    def __init__(self, env, agent_list, dict):
        assert isinstance(env, Environment), "Provided environment must be an instance of the Environment class."

        self.seed = dict['seed']
        self.epFreq = dict['recFreq']
        self.targetPath = dict['targetPath']
        self.deBug = dict['deBug']
        self.nEps = dict['nEps']
        self.env = env
        self.epLen = env.get_epLen()
        self.num_iters = dict['numIters']
        self.agent_list = agent_list
        self.data = np.zeros([dict['nEps'] * self.num_iters, 4])

        np.random.seed(self.seed)

    def run_2(self):
    
        print('Running experiment')

        for i in range(self.num_iters):
            agent = self.agent_list[i]

            for ep in range(1, self.nEps + 1):
                self.env.reset()
                oldState = self.env.state
                epReward = 0
                agent.update_policy(ep)
                pContinue = 1
                h = 0

                while pContinue > 0 and h < self.env.epLen:
                    if self.deBug:
                        print('state : ' + str(oldState))

                    action = agent.pick_action(oldState, h)

                    if self.deBug:
                        print('action : ' + str(action))

                    reward, newState, pContinue = self.env.advance(action)
                    epReward += reward

                    agent.update_obs_2(oldState, action, reward, newState, h)

                    oldState = newState
                    h += 1

                if self.deBug:
                    print('final state: ' + str(newState))
                    print('Total Reward: ' + str(epReward))

                index = ep - 1
                self.data[index, 0] = ep - 1
                self.data[index, 1] = i
                self.data[index, 2] = epReward
                self.data[index, 3] = agent.get_num_arms()

     
        print('Experiment complete')


    def save_data(self):

        print('Saving data')


        dt = pd.DataFrame(self.data, columns=['episode', 'iteration', 'epReward', 'Number_of_Balls'])
        dt = dt[(dt.T != 0).any()]
        return dt


class Node_2():
    def __init__(self, qVal, rEst, muEst, sigmaEst, num_visits, num_unique_visits,
                 num_splits, state_val, action_val, radius, action_radius):
        self.qVal = qVal
        self.rEst = rEst
        self.muEst = muEst
        self.sigmaEst = sigmaEst
        self.num_visits = num_visits
        self.num_unique_visits = num_unique_visits
        self.num_splits = num_splits
        self.state_val = state_val
        self.action_val = float(action_val)
        self.radius = radius
        self.action_radius = action_radius
        self.children = None

    def split_node_2(self, flag, epLen):
        half_radius = self.radius * 0.5
        half_action_radius = self.action_radius * 0.5
        state_val = self.state_val
        action_val = self.action_val
        num_splits_plus1 = self.num_splits + 1
        low_visits = self.num_visits <= 1

        state_offsets = [-1, 1]
        action_offsets = [-1, 1]
        children = []

        for s_off in state_offsets:
            new_state = float(state_val + s_off * half_radius)

            for a_off in action_offsets:
                new_action = float(action_val + a_off * half_action_radius)
                new_action = float(np.clip(new_action, 0, 10))

                if low_visits:
                    child = Node_2(
                        initial_q, 0, 0, 0,
                        self.num_visits, 0, num_splits_plus1,
                        new_state, new_action,
                        half_radius, half_action_radius
                    )
                else:
                    child = Node_2(
                        self.qVal, self.rEst, self.muEst, self.sigmaEst,
                        self.num_visits, self.num_visits, num_splits_plus1,
                        new_state, new_action,
                        half_radius, half_action_radius
                    )

                children.append(child)

        self.children = children
        return self.children


class Tree_2():
    def __init__(self, epLen, flag):
        self.epLen = epLen
        self.flag = flag
        self.flag_scale_2 = scaling

        self.head_1 = Node_2(initial_q, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5)
        self.head_2 = Node_2(initial_q, 0, 0, 0, 0, 0, 0, -5, 5, 5, 5)

        self.state_leaves = [self.head_1.state_val, self.head_2.state_val]
        self.vEst = [1837.1, 1837.1]
        self.tree_leaves = [self.head_1, self.head_2]

    def get_head(self):
        return self.head_1, self.head_2

    def block_diameter_2(self, node):
        return float(np.sqrt(node.radius**2 + node.action_radius**2))

    def confidence_radius_2(self, node):
        n = max(1, node.num_unique_visits)
        local_scale = 1.0 + abs(node.muEst) + np.sqrt(max(node.sigmaEst, 0.0))
        return float(self.flag_scale_2 * local_scale * np.sqrt(np.log(n + 2.0) / n))

    def should_split_2(self, node):
        if node.num_unique_visits < 2:
            return False

        conf = self.confidence_radius_2(node)
        diam = self.block_diameter_2(node)
        return conf <= diam

    def split_node_2(self, node, timestep, previous_tree):
        children = node.split_node_2(self.flag, self.epLen)

        self.tree_leaves.remove(node)
        for child in children:
            self.tree_leaves.append(child)

        child_1_state = children[0].state_val
        child_1_radius = children[0].radius

        if np.min(np.abs(np.asarray(self.state_leaves) - child_1_state)) >= child_1_radius:
            parent = node.state_val
            parent_index = self.state_leaves.index(parent)
            parent_vEst = self.vEst[parent_index]

            self.state_leaves.pop(parent_index)
            self.vEst.pop(parent_index)

            num_action_offsets = 2 ** action_dim
            self.state_leaves.append(children[0].state_val)
            self.state_leaves.append(children[num_action_offsets].state_val)
            self.vEst.append(parent_vEst)
            self.vEst.append(parent_vEst)

        return children

    def get_num_balls(self, node):
        if node.children is None:
            return 1
        num_balls = 0
        for child in node.children:
            num_balls += self.get_num_balls(child)
        return num_balls

    def get_number_of_active_balls(self):
        return self.get_num_balls(self.head_1) + self.get_num_balls(self.head_2)

    def get_active_ball_recursion(self, state, node):
        if node.children is None:
            return node, node.qVal
        else:
            active_node = None
            qVal = -np.inf

            for child in node.children:
                if self.state_within_node(state, child):
                    new_node, new_qVal = self.get_active_ball_recursion(state, child)
                    if new_qVal >= qVal:
                        active_node, qVal = new_node, new_qVal

            if active_node is None:
                return node, node.qVal

            return active_node, qVal

    def get_active_ball(self, state):
        safe_state = float(np.clip(state, -rho, rho))
        if safe_state >= 0:
            active_node, qVal = self.get_active_ball_recursion(safe_state, self.head_1)
        else:
            active_node, qVal = self.get_active_ball_recursion(safe_state, self.head_2)
        return active_node, qVal

    def state_within_node(self, state, node):
        return np.abs(state - node.state_val) <= node.radius


class AdaptiveModelBasedDiscretization_2(Agent):
    def __init__(self, epLen, numIters, scaling, split_threshold, inherit_flag, flag):
        self.epLen = epLen
        self.numIters = numIters
        self.scaling = scaling
        self.split_threshold = split_threshold
        self.inherit_flag = inherit_flag
        self.flag = flag
        self.tree_list = []

        for _ in range(epLen):
            tree = Tree_2(epLen, self.inherit_flag)
            self.tree_list.append(tree)

    def reset(self):
        self.tree_list = []
        for _ in range(self.epLen):
            tree = Tree_2(self.epLen, self.inherit_flag)
            self.tree_list.append(tree)

    def get_num_arms(self):
        total_size = 0
        for tree in self.tree_list:
            total_size += tree.get_number_of_active_balls()
        return total_size

    def update_obs_2(self, obs, action, reward, newObs, timestep):
        tree = self.tree_list[timestep]
        active_node, _ = tree.get_active_ball(obs)

        active_node.num_visits += 1
        active_node.num_unique_visits += 1
        t = active_node.num_unique_visits

        active_node.rEst = ((t - 1) * active_node.rEst + reward) / t

        if timestep != self.epLen - 1:
            delta_state = newObs - obs
            old_mu = active_node.muEst
            active_node.muEst = ((t - 1) * active_node.muEst + delta_state) / t
            active_node.sigmaEst = ((t - 1) * active_node.sigmaEst + (delta_state - old_mu) ** 2) / t

        if self.flag is False:
            if timestep == self.epLen - 1:
                active_node.qVal = min(
                    active_node.qVal,
                    initial_q,
                    active_node.rEst + self.scaling / np.sqrt(active_node.num_visits) + self.scaling * active_node.radius
                )
            else:
                next_tree = self.tree_list[timestep + 1]
                vEst = min(next_tree.vEst) + C_h * (1 + active_node.muEst ** 2 + active_node.sigmaEst ** 2)
                active_node.qVal = min(
                    active_node.qVal,
                    initial_q,
                    active_node.rEst + vEst + self.scaling / np.sqrt(active_node.num_visits) + self.scaling * active_node.radius
                )

            index = 0
            for state_val in tree.state_leaves:
                _, qMax = tree.get_active_ball(state_val)
                tree.vEst[index] = min(qMax, initial_q, tree.vEst[index])
                index += 1

        if tree.should_split_2(active_node):
            if timestep >= 1:
                _ = tree.split_node_2(active_node, timestep, self.tree_list[timestep - 1])
            else:
                _ = tree.split_node_2(active_node, timestep, None)

    def update_policy(self, k):
        if self.flag:
            for h in np.arange(self.epLen - 1, -1, -1):
                tree = self.tree_list[h]
                for node in tree.tree_leaves:
                    if node.num_unique_visits == 0:
                        node.qVal = initial_q
                    else:
                        if h == self.epLen - 1:
                            node.qVal = min(node.qVal, initial_q, node.rEst + self.scaling / np.sqrt(node.num_visits))
                        else:
                            next_tree = self.tree_list[h + 1]
                            vEst = min(next_tree.vEst) + C_h * (1 + node.muEst ** 2 + node.sigmaEst ** 2)
                            node.qVal = min(node.qVal, initial_q, node.rEst + vEst + self.scaling / np.sqrt(node.num_visits))

                index = 0
                for state_val in tree.state_leaves:
                    _, qMax = tree.get_active_ball(state_val)
                    tree.vEst[index] = min(qMax, initial_q, tree.vEst[index])
                    index += 1

    def split_ball(self, node):
        children = node.split_ball()
        return children

    def greedy(self, state, timestep, epsilon=0):
        tree = self.tree_list[timestep]
        active_node, _ = tree.get_active_ball(state)

        action = np.random.uniform(
            active_node.action_val - active_node.action_radius,
            active_node.action_val + active_node.action_radius
        )
        return float(np.clip(action, 0, 10))

    def pick_action(self, state, timestep):
        action = self.greedy(state, timestep)
        return action